# Statics — Symbolic SymPy Derivations for Combat Robotics
Every equation derived from first principles. No numerical shortcuts until the answer.

In [ ]:
from sympy import *
from IPython.display import display, Markdown

init_printing(use_latex='mathjax')

def step(label, expr=None):
    display(Markdown(f'**{label}**'))
    if expr is not None:
        display(expr)

def section(title):
    display(Markdown(f'---\n## {title}'))

## §1 — Equilibrium: The Two Laws

In [ ]:
section('Static Equilibrium')

display(Markdown(r'''
A body is in **static equilibrium** if and only if:
$$\sum \mathbf{F} = 0 \qquad \text{(net force = 0)}$$
$$\sum \mathbf{M}_O = 0 \qquad \text{(net moment about any point O = 0)}$$

In 2D this gives **3 scalar equations**: $\sum F_x=0$, $\sum F_y=0$, $\sum M_O=0$  
→ can solve for at most **3 unknowns**.
'''))

# Symbolic equilibrium setup
A, B, P, L = symbols('A B P L', positive=True)

# Simply supported beam, point load P at distance a from left
a = symbols('a', positive=True)

# Sum Fy = 0
eq_fy = Eq(A + B, P)
step('ΣF_y = 0:', eq_fy)

# Sum moments about A = 0  (eliminates A)
eq_ma = Eq(B*L, P*a)
step('ΣM_A = 0:', eq_ma)

sol = solve([eq_fy, eq_ma], [A, B])
step('Reactions:', sol)
step('B =', sol[B])
step('A =', sol[A])

# Check: if a = L/2 (center load)
step('Center load (a=L/2): A =', sol[A].subs(a, L/2), '  B =', sol[B].subs(a, L/2))

## §2 — Moment of a Force: 1 Meter Arm

In [ ]:
section('Moments and Couples')

F, d, theta = symbols('F d theta', positive=True)

# Moment = r × F
M = F * d * sin(theta)
step('Moment M = F·d·sin(θ) =', M)
step('Perpendicular case (θ=90°): M =', M.subs(theta, pi/2))

# 1 meter arm example
step('1 m arm, F=100 N, θ=90°: M =', M.subs([(F,100),(d,1),(theta,pi/2)]), 'N·m')

# Cross product form
rx, ry, Fx, Fy = symbols('r_x r_y F_x F_y', real=True)
M_cross = rx*Fy - ry*Fx
step('2D cross product: M = r_x·F_y − r_y·F_x =', M_cross)

# Couple: two equal opposite forces
display(Markdown(r'''
**Couple**: two forces $+F$ and $-F$ separated by distance $d$  
Net force = 0, but moment $M = F \cdot d$ — same about **any** point.  
Used in combat robotics: spinning weapon applies a couple to the chassis.
'''))
M_couple = F * d
step('Couple moment =', M_couple)

## §3 — Free Body Diagram: Combat Robot Wedge

In [ ]:
section('FBD: Wedge Robot Under Impact')

display(Markdown(r'''
```
         F_impact →
         ┌──────────┐
         │  WEDGE   │  W (weight down)
         │  ROBOT   │
    N1 ↑ └──────────┘ ↑ N2
    f1 →              ← f2   (friction)
    |←── L=0.4m ──────|
    A                  B
```
Impact force at height h, robot mass m, wheelbase L.
'''))

m, g_sym, h, L_wb, F_imp = symbols('m g h L F_impact', positive=True)
N1, N2 = symbols('N_1 N_2', positive=True)

W = m * g_sym

# Equilibrium equations
eq_fx = Eq(F_imp, 0)              # no horizontal acceleration in static case
eq_fy = Eq(N1 + N2, W)            # ΣFy = 0
eq_mb = Eq(N1*L_wb, W*L_wb/2 - F_imp*h)  # ΣM_B = 0

step('ΣF_y = 0:', eq_fy)
step('ΣM_B = 0 (moments about rear wheel B):', eq_mb)

sol = solve([eq_fy, eq_mb], [N1, N2])
step('Front reaction N1 =', sol[N1])
step('Rear reaction N2 =', sol[N2])
step('Impact reduces N1 (front lifts) → tipping condition: N1 = 0 when F_impact·h = W·L/2')

F_tip = solve(sol[N1], F_imp)[0]
step('Tipping force F_tip =', F_tip)

# Numeric: m=5 kg, h=0.05 m, L=0.4 m
nums = [(m,5),(g_sym,9.81),(h,Rational(5,100)),(L_wb,Rational(4,10))]
step('Example (m=5 kg, h=5 cm, L=40 cm): F_tip =',
     float(F_tip.subs(nums)), 'N')

## §4 — Truss Analysis: Method of Joints

In [ ]:
section('Truss: Method of Joints')

display(Markdown(r'''
Simple triangular truss (combat robot frame cross-section):
```
    C
   /|\
  / | \
 /  |  \
A───┼───B
↑Ra     ↑Rb
        P↓ (at C)
```
Members: AC, BC, AB. External load P downward at C.
'''))

P_load, L_span, H = symbols('P L H', positive=True)
F_AC, F_BC, F_AB = symbols('F_AC F_BC F_AB', real=True)

# Geometry: A at (0,0), B at (L,0), C at (L/2, H)
theta_AC = atan(H / (L_span/2))   # angle of AC from horizontal
step('Angle of members from horizontal θ =', theta_AC)

# Support reactions first
Ra, Rb = symbols('R_A R_B', positive=True)
eq1 = Eq(Ra + Rb, P_load)
eq2 = Eq(Rb * L_span, P_load * L_span/2)   # moment about A
rxns = solve([eq1, eq2], [Ra, Rb])
step('Support reactions:', rxns)

# Joint A: ΣFx=0, ΣFy=0
eqAx = Eq(F_AC*cos(theta_AC) + F_AB, 0)
eqAy = Eq(F_AC*sin(theta_AC) + rxns[Ra], 0)
sol_A = solve([eqAx, eqAy], [F_AC, F_AB])
step('Member AC (+ = tension):', simplify(sol_A[F_AC]))
step('Member AB:', simplify(sol_A[F_AB]))
step('Negative → compression (strut), positive → tension (tie)')

## §5 — Nylon Material Properties for Combat Robots

In [ ]:
section('Nylon vs Steel: Stress and Safety Factor')

display(Markdown(r'''
| Property | Nylon 6/6 | 6061-T6 Al | 4140 Steel |
|---|---|---|---|
| Density (g/cm³) | 1.14 | 2.70 | 7.85 |
| Tensile strength (MPa) | 80 | 310 | 1020 |
| Yield strength (MPa) | 55 | 276 | 655 |
| Elastic modulus (GPa) | 2.7 | 68.9 | 200 |
| Impact strength (J/m) | 53 | 15 | 54 |
| Cost | low | medium | high |

**Nylon wins** on: weight, machinability, impact absorption (energy goes into deformation not fracture)  
**Nylon loses** on: stiffness (10× less than Al), creep under sustained load, moisture absorption
'''))

# Axial stress and safety factor
F_axial, A_cross, sigma_y, SF = symbols('F A sigma_y SF', positive=True)
sigma = F_axial / A_cross
step('Axial stress σ = F/A =', sigma)

SF_expr = sigma_y / sigma
step('Safety factor SF = σ_y / σ =', SF_expr)

# Nylon bracket: 80N load on 10×5mm cross section
F_n = 80          # N
A_n = 10e-3*5e-3  # m²
sy_nylon = 55e6   # Pa
sigma_n = F_n / A_n
SF_n = sy_nylon / sigma_n
step(f'Nylon bracket: σ = {sigma_n/1e6:.2f} MPa,  SF = {SF_n:.1f}')

# Deflection of a beam
E_mod, I_moment, P_b, x, L_b = symbols('E I P x L', positive=True)
# Simply supported, center load: δ_max = PL³/(48EI)
delta_max = P_b * L_b**3 / (48 * E_mod * I_moment)
step('Max deflection (center load, simply supported):', delta_max)

# Rectangular cross section I = bh³/12
b_w, h_w = symbols('b h', positive=True)
I_rect = b_w * h_w**3 / 12
step('Rectangular section I = bh³/12 =', I_rect)
step('→ doubling height h reduces deflection by 8×')

## §6 — Spinning Weapon: Angular Momentum and Impact Energy

In [ ]:
section('Weapon Kinetics')

I_disk, omega_w, m_w, r_w, v_tip = symbols('I omega m r v_tip', positive=True)

# Disk moment of inertia
I_disk_expr = Rational(1,2) * m_w * r_w**2
step('Disk moment of inertia I = ½mr² =', I_disk_expr)

# Rotational KE
KE = Rational(1,2) * I_disk_expr * omega_w**2
step('Kinetic energy KE = ½Iω² =', KE)
step('= ¼mr²ω² =', simplify(KE))

# Tip speed
v_tip_expr = r_w * omega_w
step('Tip speed v_tip = rω =', v_tip_expr)

# Express KE in terms of tip speed
KE_v = KE.subs(omega_w, v_tip/r_w)
step('KE in terms of tip speed:', simplify(KE_v))
step('= ¼mv_tip²  →  KE scales with tip speed², not RPM directly')

# Numeric: BattleBots-scale (3 lb = 1.36 kg, r=0.1m, 5000 RPM)
import math
m_num   = 1.36
r_num   = 0.10
rpm     = 5000
omega_n = rpm * 2*math.pi / 60
KE_num  = 0.25 * m_num * r_num**2 * omega_n**2
v_num   = r_num * omega_n
step(f'3 lb disk, r=10 cm, 5000 RPM:')
print(f'  ω = {omega_n:.1f} rad/s')
print(f'  v_tip = {v_num:.1f} m/s = {v_num*3.28:.0f} ft/s')
print(f'  KE = {KE_num:.1f} J')

# Angular impulse on chassis
display(Markdown(r'''
**Gyroscopic effect**: spinning weapon resists turning of chassis.
$$\tau_{gyro} = \mathbf{\omega}_{chassis} \times \mathbf{L}_{weapon} = \Omega \cdot I \cdot \omega_w$$
Fast turns at high weapon speed → gyroscopic torque tips the robot.
'''))
Omega_turn = symbols('Omega_turn', positive=True)
tau_gyro = Omega_turn * I_disk_expr * omega_w
step('Gyroscopic torque τ = Ω·I·ω =', tau_gyro)

## §7 — Friction, Traction, and Drive Train

In [ ]:
section('Friction and Drive')

mu, N_normal, F_drive, m_robot, g_sym = symbols('mu N F_drive m g', positive=True)

# Max traction force
F_trac = mu * N_normal
step('Max traction force F_trac = μN =', F_trac)
step('For flat surface: N = mg, so F_trac = μmg')

# Typical μ values
display(Markdown(r'''
| Surface | μ (rubber wheel) |
|---|---|
| BattleBox steel plate | 0.6–0.8 |
| Polycarbonate arena | 0.5–0.7 |
| Wet/oil contaminated | 0.2–0.3 |
'''))

# Acceleration limit from traction
a_max = mu * g_sym
step('Max acceleration (traction limited): a_max = μg =', a_max)
step('At μ=0.7, g=9.81: a_max =', float(0.7*9.81), 'm/s²')

# Motor torque → wheel force
tau_m, r_wheel, eta_drive = symbols('tau r eta', positive=True)
F_wheel = eta_drive * tau_m / r_wheel
step('Wheel force from motor torque: F = η·τ/r =', F_wheel)
step('η = drivetrain efficiency (0.85 typical for chain/belt)')

# Required motor torque to spin-up weapon
t_spinup, I_weapon, omega_final = symbols('t_spinup I_w omega_f', positive=True)
alpha = omega_final / t_spinup
tau_spinup = I_weapon * alpha
step('Weapon spin-up torque τ = I·α = I·ω_f/t =', tau_spinup)

# Numeric: disk KE=100J in 3 seconds
# I=0.0068 kg·m², ω=500 rad/s in 3s
print(f'Example: I=0.0068 kg·m², ω=500 rad/s, t=3s')
print(f'  τ_spinup = {0.0068*500/3:.2f} N·m')
print(f'  KE stored = {0.5*0.0068*500**2:.0f} J')

## §8 — Machining Tolerances: Fit and Clearance

In [ ]:
section('Tolerances and Fits')

display(Markdown(r'''
**ISO fit system** — shaft/hole clearance for bearings and shafts:

| Fit type | Example | Use case |
|---|---|---|
| Clearance | H7/g6 | running bearing, easy assembly |
| Transition | H7/k6 | locating fit, light press |
| Interference | H7/p6 | press fit, permanent assembly |

For a **20 mm shaft**:
- H7 hole: +0.021 / 0 mm  (20.000 – 20.021 mm)
- g6 shaft: −0.007 / −0.020 mm  (19.980 – 19.993 mm)
- Clearance: 0.007 to 0.041 mm
'''))

# Tolerance stack-up
t1, t2, t3 = symbols('t_1 t_2 t_3', positive=True)
T_stack = sqrt(t1**2 + t2**2 + t3**2)   # RSS (statistical)
T_worst = t1 + t2 + t3                   # worst case
step('Worst-case stack-up T =', T_worst)
step('RSS (statistical) stack-up T =', T_stack)

# Numeric: three 0.1mm tolerances
step(f'Three ±0.1mm tolerances:')
print(f'  Worst case: ±{3*0.1:.1f} mm')
print(f'  RSS:        ±{(3*(0.1**2))**0.5:.3f} mm')

display(Markdown(r'''
**Nylon machining notes**:
- Cut dry or with compressed air (coolant swells nylon)
- Sharp tools, high RPM, slow feed → clean cuts
- Nylon absorbs ~2.5% moisture → dimensions change ±0.2% after machining
- Leave 0.1–0.2 mm on interference fits to account for moisture expansion
- Drill undersized, ream to final diameter for precision holes
'''))